In [1]:
#converting long dataset into chunks (enough samples for model to get trained)
#model used:DNABERT 2 model

In [2]:
pip install torch torchvision torchaudio transformers datasets accelerate sentencepiece protobuf einops triton scikit-learn pandas numpy biopython scipy safetensors huggingface_hub tokenizers

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement triton (from versions: none)

[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
ERROR: No matching distribution found for triton


In [3]:
pip install --upgrade transformers

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
pip install torch torchvision torchaudio sentencepiece accelerate einops protobuf

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os

os.environ["FLASH_ATTENTION_FORCE_DISABLE"] = "1"
os.environ["USE_FLASH_ATTENTION"] = "0"
os.environ["TRITON_DISABLED"] = "1"

In [6]:
#storing sequences
from Bio import SeqIO
def read_fasta(file):
    return [str(record.seq) for record in SeqIO.parse(file,"fasta")]
meca_seq=read_fasta(r"D:\c programming\ibs2\mrsaproject\mrsa-vs-nonmrsa-dnabert\datasets\mecA_complete_clean.fasta")
mecc_seq=read_fasta(r"D:\c programming\ibs2\mrsaproject\mrsa-vs-nonmrsa-dnabert\datasets\mecC_clean.fasta")
mssa_seq=read_fasta(r"D:\c programming\ibs2\mrsaproject\mrsa-vs-nonmrsa-dnabert\datasets\mssa_clean.fasta")

In [7]:
#split into chunks
def split_seq(seq,chunk_size=300):
    return [
        seq[i:i+chunk_size] for i in range(0,len(seq),chunk_size) if(len(seq[i:i+chunk_size])==chunk_size)
    ]


In [8]:
meca_chunks=[]
for seq in meca_seq: 
   meca_chunks.extend(split_seq(seq))

mecc_chunks=[]
for seq in mecc_seq: 
   mecc_chunks.extend(split_seq(seq))

mssa_chunks=[]
for seq in mssa_seq: 
   mssa_chunks.extend(split_seq(seq))
print(len(meca_chunks[0]))

300


In [9]:
#creating labels
import pandas as pd
mrsa_chunks=meca_chunks+mecc_chunks
df=pd.DataFrame({
    "sequence":mrsa_chunks+mssa_chunks,
    "label":[1]*len(mrsa_chunks)+[0]*len(mssa_chunks)
})


In [10]:
#balancing dataset
min_size=min(len(mrsa_chunks),len(mssa_chunks))

df_mrsa=df[df.label==1].sample(min_size)
df_mssa=df[df.label==0].sample(min_size)
#mixes random rows together and then reset index
df=pd.concat([df_mrsa,df_mssa]).sample(frac=1).reset_index(drop=True)


In [11]:
#creating the dataset and then splitting into test and training part 
from datasets import Dataset
dataset=Dataset.from_pandas(df)
dataset=dataset.train_test_split(test_size=0.2)

In [12]:
#making kmers 
def to_kmers(seq,k=6):
    return "".join([seq[i:i+k] for i in range (len(seq)-k+1)])
#sequence field is now replaced by kmers of len 6
dataset=dataset.map(lambda x:{"sequence":to_kmers(x["sequence"])})


Map:   0%|          | 0/436947 [00:00<?, ? examples/s]

Map:   0%|          | 0/109237 [00:00<?, ? examples/s]

In [15]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "zhihan1996/DNA_bert_6"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

config.json: 0.00B [00:00, ?B/s]

D:\c programming\python\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\yasha\.cache\huggingface\hub\models--zhihan1996--DNA_bert_6. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/40.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/359M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: zhihan1996/DNA_bert_6
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/359M [00:00<?, ?B/s]

In [16]:
def tokenize(example):
    return tokenizer(
        example["sequence"], #input data
        truncation=True,#if input text>limit cut off tokens
        padding="max_length",#every output of eq len
        max_length=512
    )

In [17]:
#removing unused text column
dataset=dataset.remove_columns(["sequence"])
dataset.set_format("torch")

In [18]:
from transformers import TrainingArguments
training_args=TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    eval_strategy="epoch",
    logging_dir="./logs"
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [19]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

In [20]:

from transformers import Trainer
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"]
)